In [25]:
import os
import pandas as pd
from bs4 import BeautifulSoup


In [26]:
import sys
print("Python:", sys.executable)

try:
    import html5lib
    print("✓ html5lib imported successfully")
except ImportError as e:
    print("✗ html5lib import failed:", e)

try:
    import bs4
    print("✓ beautifulsoup4 imported successfully")
except ImportError as e:
    print("✗ beautifulsoup4 import failed:", e)

try:
    import lxml
    print("✓ lxml imported successfully")
except ImportError as e:
    print("✗ lxml import failed:", e)

Python: /Library/Developer/CommandLineTools/usr/bin/python3
✓ html5lib imported successfully
✓ beautifulsoup4 imported successfully
✓ lxml imported successfully


In [27]:
#creating list of all boxscores 
SCORE_DIR = "data/scores"
box_scores = os.listdir(SCORE_DIR)
box_scores = [os.path.join(SCORE_DIR,f) for f in box_scores if f.endswith(".html")]

In [28]:
#take in one boxscore and clean up the html
def parse_html(box_score):
    with open(box_score) as f:
        html = f.read()
    soup = BeautifulSoup(html)
    [s.decompose() for s in soup.select("tr.over_header")]
    [s.decompose() for s in soup.select("tr.thead")]
    return soup

In [29]:
def read_line_score(soup):
    #reading in html
    try:
        line_score = pd.read_html(str(soup),attrs={"id":"line_score"})[0]
    except ValueError:
        print("trouble here!")
        raise
    cols = list(line_score.columns)
    cols[0] = "team"
    cols[-1] = "total"
    line_score.columns = cols

    #removing quarters from table
    line_score = line_score[["team","total"]]
    return line_score

In [30]:
def read_stats(soup,team,stat):
    df = pd.read_html(str(soup), attrs = {'id': f'box-{team}-game-{stat}'}, index_col=0)[0]
    df = df.apply(pd.to_numeric,errors="coerce")
    return df
     

In [31]:
#read season info at bottom of page to find season date
def read_season_info(soup):
    nav = soup.select("#bottom_nav_container")[0]
    hrefs = [a["href"] for a in nav.find_all('a')]
    season = os.path.basename(hrefs[1]).split("_")[0]
    return season


In [32]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

In [33]:
#for each boc score we will have 2 teams that share one boxscore
#each team will have one basic and one advanced stats
#summary will contain the max basic and max of each team
#summaries will have both
#and then after the loop summary will be turned into a df containing info on both teams 
base_cols = None
games = []
for box_score in box_scores:
    try:
        soup = parse_html(box_score)
        line_score = read_line_score(soup)
        teams = list(line_score["team"])

        summaries = []
        for team in teams:
            basic = read_stats(soup,team,"basic")
            advanced = read_stats(soup,team,"advanced")

            #total stats of team
            totals = pd.concat([basic.iloc[-1,:],advanced.iloc[-1,:]])
            totals.index = totals.index.str.lower()

            maxes = pd.concat([basic.iloc[:-1,:].max(),advanced.iloc[:-1,:].max()])
            maxes.index = maxes.index.str.lower() +"_max"

            summary = pd.concat([totals,maxes])

            #removes duplicates and bpm and only keeps the ones seen first
            if base_cols is None:
                base_cols = list(summary.index.drop_duplicates(keep="first"))
                base_cols = [b for b in base_cols if "bpm" not in b]

            summary = summary[base_cols]

            #adding summaries of both teams to summary
            summaries.append(summary)

        summary = pd.concat(summaries,axis=1).T

        #add line score
        game = pd.concat([summary,line_score],axis=1)

        #assign home and away team
        game["home"] = [0,1]

        #create info of the homes opp
        game_opp = game.iloc[::-1].reset_index()
        game_opp.columns += "_opp"

        full_game = pd.concat([game,game_opp],axis=1)

        full_game["season"] = read_season_info(soup)
        full_game["date"] = os.path.basename(box_score)[:8] #first 8 chars are the date
        full_game["date"] = pd.to_datetime(full_game["date"],format="%Y%m%d") 

        full_game["won"] =  full_game["total"] > full_game["total_opp"]
        games.append(full_game)

        #stats of loop
        if len(games) %100 == 0:
            print(f"{len(games)} / {len(box_scores)}")
    except Exception as e:
        continue



100 / 7094
200 / 7094
300 / 7094
400 / 7094
500 / 7094
trouble here!
600 / 7094
trouble here!
700 / 7094
800 / 7094
900 / 7094
trouble here!
trouble here!
1000 / 7094
1100 / 7094
1200 / 7094
1300 / 7094
1400 / 7094
1500 / 7094
1600 / 7094
1700 / 7094
trouble here!
1800 / 7094
1900 / 7094
2000 / 7094
2100 / 7094
2200 / 7094
trouble here!
2300 / 7094
2400 / 7094
2500 / 7094
trouble here!
2600 / 7094
2700 / 7094
2800 / 7094
2900 / 7094
3000 / 7094
3100 / 7094
3200 / 7094
3300 / 7094
3400 / 7094
3500 / 7094
trouble here!
3600 / 7094
3700 / 7094
3800 / 7094
3900 / 7094
4000 / 7094
4100 / 7094
trouble here!
4200 / 7094
4300 / 7094
4400 / 7094
4500 / 7094
4600 / 7094
4700 / 7094
trouble here!
4800 / 7094
4900 / 7094
5000 / 7094
5100 / 7094
5200 / 7094
trouble here!
5300 / 7094
5400 / 7094
5500 / 7094
5600 / 7094
5700 / 7094
5800 / 7094
5900 / 7094
6000 / 7094
6100 / 7094
6200 / 7094
6300 / 7094
6400 / 7094
6500 / 7094
6600 / 7094
trouble here!
6700 / 7094
6800 / 7094
6900 / 7094
7000 / 7094


In [39]:
len(games)



7082

In [35]:
games_df = pd.concat(games,ignore_index=True)

In [36]:
games_df.dtypes


mp                  float64
mp                  float64
fg                  float64
fga                 float64
fg%                 float64
                  ...      
total_opp             int64
home_opp              int64
season               object
date         datetime64[ns]
won                    bool
Length: 154, dtype: object

In [37]:
games_df.to_csv("nba_games_2019to2025.csv")

In [38]:
import sys
print(sys.executable)

/Library/Developer/CommandLineTools/usr/bin/python3
